# LangChain

In [35]:
import os
from pathlib import Path
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")

In [7]:
from langchain_gigachat import GigaChat
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    temperature=0.1
)

In [ ]:
messages = [
    ("system", "You are an expert in {domain}. Your task is answer the question as short as possible"),
    MessagesPlaceholder("history"),
]
prompt_template = ChatPromptTemplate(messages)


domain = input('Choice domain area: ')
history = []
while True:
    print()
    user_content = input('You: ')
    history.append(HumanMessage(content=user_content))
    prompt_value = prompt_template.invoke({"domain": domain, "history": history})
    full_ai_content = ""
    print('Bot: ', end="")
    for ai_message_chunk in llm.stream(prompt_value.to_messages()):
        print(ai_message_chunk.content, end="")
        full_ai_content += ai_message_chunk.content
    history.append(AIMessage(content=full_ai_content))
    print()


Bot: Здравствуйте! Как я могу Вам помочь?

Bot: Привет! Рад тебя видеть. Как твои дела?

Bot: Здорово, что тебе интересно! О чем хочешь поговорить?

Bot: О чём хочешь узнать или обсудить?



# Output Parser

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage


output_parser = StrOutputParser()
answer = AIMessage(content="Cats are beautiful")

print(output_parser.invoke(answer))
# "Cats are beautiful"

Cats are beautiful


In [25]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class Person(BaseModel):
    firstname: str = Field(validation_alias="firstName", description="firstname of hero")
    lastname: str = Field(validation_alias="lastName", description="lastname of hero")
    age: int = Field(validation_alias="age", description="age of hero")


output_parser = PydanticOutputParser(pydantic_object=Person)
answer = AIMessage(content='{"firstname": "John", "lastname": "Smith", "age": 45}')
print(output_parser.invoke(answer))
# Person(firstname='John' lastname='Smith' age=45)

OutputParserException: Failed to parse Person from completion {"firstname": "John", "lastname": "Smith", "age": 45}. Got: 2 validation errors for Person
firstName
  Field required [type=missing, input_value={'firstname': 'John', 'la...me': 'Smith', 'age': 45}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
lastName
  Field required [type=missing, input_value={'firstname': 'John', 'la...me': 'Smith', 'age': 45}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [31]:
llm = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    temperature=0
)

messages = [
    ("system", "Handle the user query.\n{format_instructions}. Display only first output!"),
    ("human", "{user_query}")
]
prompt_template = ChatPromptTemplate(messages)

prompt_value = prompt_template.invoke(
    {
        "format_instructions": output_parser.get_format_instructions(),
        "user_query": "Генрих Смит был восемнацдцателетним юношей, мечтающим уехать в город"
    }
)

answer = llm.invoke(prompt_value.to_messages())
print(output_parser.invoke(answer))
# Person(firstname='Генрих', lastname='Смит', age=18)

firstname='Генрих' lastname='Смит' age=12


# Runnable

In [3]:
from langchain_core.runnables import RunnableLambda

square_runnable = RunnableLambda(lambda x: x ** 2)
result = square_runnable.invoke(10)
print(result) # 100

100


In [4]:
import math
from langchain_core.runnables import RunnableSequence

square_runnable = RunnableLambda(lambda x: x ** 2)
add_10_runnable = RunnableLambda(lambda x: x + 10)
log_runnable = RunnableLambda(lambda x: math.log(x))

pipeline = RunnableSequence(square_runnable, add_10_runnable, log_runnable)
result = pipeline.invoke(10)
print(result) # ~ 4.7

4.700480365792417


In [5]:
pipeline = square_runnable | add_10_runnable | log_runnable
result = pipeline.invoke(10)
print(result) # ~ 4.7

4.700480365792417


In [6]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

square_runnable = RunnableLambda(lambda x: x ** 2)
add_10_runnable = RunnableLambda(lambda x: x + 10)

chain = RunnableParallel(square_result=square_runnable, add_10_result=add_10_runnable)

result = chain.invoke(2)
print(result) # {'sqare_result': 4, 'add_10_result': 12}

{'square_result': 4, 'add_10_result': 12}


In [2]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

runnable_1 = RunnableLambda(lambda x: x + "+" + "1")
runnable_2 = RunnableLambda(lambda x: x + "+" + "2")
runnable_3 = RunnableLambda(lambda x: x + "+" + "3")
runnable_4 = RunnableLambda(lambda x: '[' + x['some'] + ']+[' + x['other'] + ']')

chain = runnable_1 | {'some': runnable_2, 'other': runnable_3} | runnable_4

res = chain.invoke("0")

res

'[0+1+2]+[0+1+3]'

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

square_runnable = RunnableLambda(lambda data: data["initial_value"] ** 2)

chain = RunnablePassthrough.assign(square_result=square_runnable)

res = chain.invoke({"initial_value": 2})
print(res) # {'initial_value': 2, 'square_result': 4}

{'initial_value': 2, 'square_result': 4}


In [ ]:
calc_discriminant = RunnablePassthrough.assign(discriminant=RunnableLambda(lambda args: args['b'] ** 2 - 4 * args['a'] * args['c']))
calc_roots = RunnableLambda(lambda data: {'x': data['b'] / (2 * data['a'])} \
                            if data['discriminant'] == 0 else {'x1': (-data['b'] + data['discriminant']**0.5) / (2 * data['a']), 'x2': (-data['b'] - data['discriminant']**0.5) / (2 * data['a'])} \
                                if data['discriminant'] > 0 else {'message': 'Корней нет'})

chain = calc_discriminant | calc_roots
res = chain.invoke({'a': 2, "b": 4, "c": 1})
res

{'message': 'Корней нет'}

# LCEL

In [30]:
from langchain_gigachat import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate(
    [('system', "You are a cat."),
    ('user', "My request: {user_query}.")]
)

chat_model = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
)

output_parser = StrOutputParser()

chain = prompt | chat_model | output_parser

chain.invoke({"user_query": "Hi, AI!"})

'Meow! Hello there, human friend. What purr-ty can I help you with today? 😊'

In [31]:
chain = prompt | chat_model | (lambda message: message.content)

chain.invoke({"user_query": "Что делаешь?"})

'Я лежу на подоконнике, греюсь в лучах солнца и лениво наблюдаю за птичками снаружи. Время от времени лениво потягиваюсь и ловлю солнечный зайчик лапкой. Жизнь прекрасна!'

# Advanced Message Processing

In [32]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, trim_messages


messages = [
    SystemMessage("Ты добрый дворецкий"),
    HumanMessage("Доброе утро!"),
    AIMessage("Здравствуйте!"),
    HumanMessage("Как Ваши дела?"),
    AIMessage("Неплохо! А у вас как?"),
    HumanMessage("Тоже неплохо! Как вы поживаете, как здоровье?"),
    AIMessage("Отлично! Спасибо, что поинтересовались."),
    HumanMessage("Ну, право. До свидания, рад был повидаться!"),
    AIMessage("Взаимно, взаимно. До встречи."),
]

trimmer = trim_messages(
    strategy="last",
    token_counter=len,
    max_tokens=6,
    start_on="human",
    end_on="human",
    include_system=True,
    allow_partial=False
)

new_messages = trimmer.invoke(messages)
print(new_messages)

[SystemMessage(content='Ты добрый дворецкий', additional_kwargs={}, response_metadata={}), HumanMessage(content='Как Ваши дела?', additional_kwargs={}, response_metadata={}), AIMessage(content='Неплохо! А у вас как?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Тоже неплохо! Как вы поживаете, как здоровье?', additional_kwargs={}, response_metadata={}), AIMessage(content='Отлично! Спасибо, что поинтересовались.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Ну, право. До свидания, рад был повидаться!', additional_kwargs={}, response_metadata={})]


In [34]:
import time

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, trim_messages

DEFAULT_SESSION_ID = "default"
chat_history = InMemoryChatMessageHistory()

trimmer = trim_messages(
    strategy="last",
    token_counter=len,
    max_tokens=6,
    start_on="human",
    end_on="human",
    include_system=True,
    allow_partial=False
)

chain = trimmer | chat_model
chain_with_history = RunnableWithMessageHistory(chain, lambda session_id: chat_history)

chain_with_history.invoke(
    [HumanMessage("Hi, my name is Bob!")],
    config={"configurable": {"session_id": DEFAULT_SESSION_ID}},
)
ai_message = chain_with_history.invoke(
    [HumanMessage("What is my name?")],
    config={"configurable": {"session_id": DEFAULT_SESSION_ID}},
)

print(ai_message.content)

Твоё имя — Боб.
